In [1]:
from pathlib import Path
import sys
import torch


def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "AAAI24_GARCH_NN_Reproduction").exists() and (c / "dataset").exists():
            return c
    raise RuntimeError("Cannot locate project root from current working directory")


def describe_gpu_and_pick_device():
    if not torch.cuda.is_available():
        print("CUDA available: False")
        return "cpu"

    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    total_gb = props.total_memory / (1024 ** 3)
    print("CUDA available: True")
    print(f"GPU: {props.name} (index={device_idx})")
    print(f"Total VRAM: {total_gb:.2f} GB")
    print(f"CUDA capability: {props.major}.{props.minor}")
    return f"cuda:{device_idx}"


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEFAULT_DEVICE = describe_gpu_and_pick_device()

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Selected device: {DEFAULT_DEVICE}")

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU (index=0)
Total VRAM: 4.00 GB
CUDA capability: 8.6
Project root: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT
Python executable: d:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\.venv\Scripts\python.exe
Selected device: cuda:0


In [3]:
DATASET_DIR = PROJECT_ROOT / "dataset"
DATASET_NAMES = ["VN30_INDEX.csv", "DAX_40.csv", "EuroNext_100.csv", "IBEX_35.csv", "KOSPI_index.csv", "Nikkei_225.csv", "SMI.csv", "snp500.csv", "VN_INDEX.csv"]

# Keep order from DATASET_NAMES and validate each file explicitly.
DATASET_FILES = [DATASET_DIR / name for name in DATASET_NAMES]
missing_files = [str(p) for p in DATASET_FILES if not p.exists()]
assert not missing_files, f"Dataset file(s) not found: {missing_files}"

print(f"Dataset dir: {DATASET_DIR}")
print(f"Selected {len(DATASET_FILES)} dataset file(s):")
for p in DATASET_FILES:
    print(" -", p.name)

Dataset dir: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\dataset
Selected 9 dataset file(s):
 - VN30_INDEX.csv
 - DAX_40.csv
 - EuroNext_100.csv
 - IBEX_35.csv
 - KOSPI_index.csv
 - Nikkei_225.csv
 - SMI.csv
 - snp500.csv
 - VN_INDEX.csv


In [ ]:
import importlib
import AAAI24_GARCH_NN_Reproduction.experiments.run_benchmark as run_benchmark_module

run_benchmark_module = importlib.reload(run_benchmark_module)
run_benchmark = run_benchmark_module.run_benchmark

SEQ_LEN = 60
EPOCHS = 60
BATCH_SIZE = 64
DEVICE = DEFAULT_DEVICE
NUM_WORKERS = 2 if str(DEVICE).startswith("cuda") else 0
LOG_PROGRESS = True

if str(DEVICE).startswith("cuda"):
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

RESULTS_DIR = PROJECT_ROOT / "AAAI24_GARCH_NN_Reproduction" / "experiments" / "results_4"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

#OUTPUT_CSV = RESULTS_DIR / "model_results_8_1_1.csv"   
#OUTPUT_PREDICTIONS_CSV = RESULTS_DIR / "predictions_8_1_1.csv"

OUTPUT_CSV = RESULTS_DIR / "model_results_fixed_split_data.csv"
OUTPUT_PREDICTIONS_CSV = RESULTS_DIR / "predictions.csv"

print(f"Dataset dir: {DATASET_DIR}")
print(f"Dataset files ({len(DATASET_FILES)}):")
for p in DATASET_FILES:
    print(" -", p.name)
print(f"Seq len: {SEQ_LEN}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {DEVICE}")
print(f"Num workers: {NUM_WORKERS}")
print(f"Log progress: {LOG_PROGRESS}")
print(f"Output metrics CSV: {OUTPUT_CSV}")
print(f"Output predictions CSV: {OUTPUT_PREDICTIONS_CSV}")

Dataset dir: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\dataset
Dataset files (9):
 - VN30_INDEX.csv
 - DAX_40.csv
 - EuroNext_100.csv
 - IBEX_35.csv
 - KOSPI_index.csv
 - Nikkei_225.csv
 - SMI.csv
 - snp500.csv
 - VN_INDEX.csv
Seq len: 60
Epochs: 60
Batch size: 64
Device: cuda:0
Num workers: 2
Log progress: True
Output metrics CSV: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\AAAI24_GARCH_NN_Reproduction\experiments\results_3\model_results_fixed_split_data.csv
Output predictions CSV: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\AAAI24_GARCH_NN_Reproduction\experiments\results_3\predictions_fixed_split_data.csv


In [5]:
# Data Processing and Splitting Configuration
# ================================================

# Split mode: "ratio" (8:1:1 proportional split) or "fixed_counts" (predefined table for 2010-2025)
SPLIT_MODE = "fixed_counts"  # Change to "ratio" to use 8:1:1 proportional split

# Date range filtering (only used for filtering, not in ratio mode)
# If DATE_START/DATE_END are None, no filtering is applied (uses all available data)
DATE_START = "2010-01-01"  # Format: YYYY-MM-DD or None
DATE_END = "2025-12-31"    # Format: YYYY-MM-DD or None

print(f"Data Split Configuration:")
print(f"  Split mode: {SPLIT_MODE}")
print(f"  Date range: {DATE_START or 'from start'} to {DATE_END or 'to end'}")
if SPLIT_MODE == "fixed_counts":
    print(f"  (Using predefined Train/Val/Test counts from FIXED_SPLITS table)")
elif SPLIT_MODE == "ratio":
    print(f"  (Using 80%-10%-10% proportional split)")

Data Split Configuration:
  Split mode: fixed_counts
  Date range: 2010-01-01 to 2025-12-31
  (Using predefined Train/Val/Test counts from FIXED_SPLITS table)


In [6]:
# Inspect Train/Val/Test Splits For All Datasets Before Running Pipeline
# ======================================================================
# This cell validates split quality for every dataset in DATASET_FILES.

from AAAI24_GARCH_NN_Reproduction.core.data_processor import (
    load_close_series,
    prepare_aaai24_data,
    FIXED_SPLITS,
    DEFAULT_VOL_WINDOW,
    prepare_aaai24_series,
    filter_close_by_date,
    _normalize_dataset_name,
    DEFAULT_SEQ_LEN,
    create_sliding_windows,
)
import pandas as pd

rows = []
print(f"Checking {len(DATASET_FILES)} dataset(s)...")
print("=" * 90)

for dataset_file in DATASET_FILES:
    dataset_name = dataset_file.stem
    close = load_close_series(dataset_file)

    close_filtered = filter_close_by_date(close, DATE_START, DATE_END, verbose=False)
    returns_pre, vol_pre = prepare_aaai24_series(
        close_filtered,
        volatility_window=DEFAULT_VOL_WINDOW,
        verbose=False,
    )
    expected_after_preprocess = len(returns_pre)

    train_split, val_split, test_split = prepare_aaai24_data(
        close,
        split_mode=SPLIT_MODE,
        dataset_name=dataset_name,
        date_start=DATE_START,
        date_end=DATE_END,
        verbose=False,
    )

    train_r, train_v = train_split
    val_r, val_v = val_split
    test_r, test_v = test_split

    train_n = len(train_r)
    val_n = len(val_r)
    test_n = len(test_r)
    actual_total = train_n + val_n + test_n

    expected_train = None
    expected_val = None
    expected_test = None
    expected_total_raw = None
    expected_total_after_vol = None
    expected_test_after_vol = None
    counts_match_after_vol = None

    if SPLIT_MODE == "fixed_counts":
        norm_name = _normalize_dataset_name(dataset_name)
        if norm_name in FIXED_SPLITS:
            expected_train, expected_val, expected_test = FIXED_SPLITS[norm_name]
            expected_total_raw = expected_train + expected_val + expected_test
            expected_total_after_vol = max(expected_total_raw - DEFAULT_VOL_WINDOW, 0)
            expected_test_after_vol = max(expected_test - DEFAULT_VOL_WINDOW, 0)
            counts_match_after_vol = (
                train_n == expected_train
                and val_n == expected_val
                and test_n == expected_test_after_vol
            )
    elif SPLIT_MODE == "ratio":
        n = expected_after_preprocess
        expected_train = int(n * 0.8)
        expected_val = int(n * 0.9) - expected_train
        expected_test = n - int(n * 0.9)
        expected_total_raw = expected_train + expected_val + expected_test
        expected_total_after_vol = expected_total_raw
        expected_test_after_vol = expected_test
        counts_match_after_vol = (
            train_n == expected_train
            and val_n == expected_val
            and test_n == expected_test_after_vol
        )

    nan_issues = int(
        train_r.isna().sum()
        + train_v.isna().sum()
        + val_r.isna().sum()
        + val_v.isna().sum()
        + test_r.isna().sum()
        + test_v.isna().sum()
    )

    window_issues = []
    for split_name, r_split, v_split in [
        ("Train", train_r, train_v),
        ("Val", val_r, val_v),
        ("Test", test_r, test_v),
    ]:
        try:
            create_sliding_windows(r_split, v_split, seq_len=DEFAULT_SEQ_LEN, horizon=1)
        except ValueError as e:
            window_issues.append(f"{split_name}: {e}")

    rows.append(
        {
            "Dataset": dataset_name,
            "OriginalClosePoints": len(close),
            "AfterDateFilterPoints": len(close_filtered),
            "ExpectedAfterPreprocess": expected_after_preprocess,
            "Train": train_n,
            "Val": val_n,
            "Test": test_n,
            "ActualTotal": actual_total,
            "ExpectedTrain": expected_train,
            "ExpectedVal": expected_val,
            "ExpectedTestRaw": expected_test,
            "ExpectedTestAfterVol": expected_test_after_vol,
            "ExpectedTotalRaw": expected_total_raw,
            "ExpectedTotalAfterVol": expected_total_after_vol,
            "TotalGapAfterVol": (expected_total_after_vol - actual_total) if expected_total_after_vol is not None else None,
            "CountsMatchAfterVol": counts_match_after_vol,
            "NaNIssues": nan_issues,
            "WindowCheckOK": len(window_issues) == 0,
            "WindowIssues": " | ".join(window_issues),
        }
    )

summary_df = pd.DataFrame(rows).sort_values("Dataset").reset_index(drop=True)

print("Split summary across all datasets:")
display(summary_df)

problem_mask = (
    (summary_df["NaNIssues"] > 0)
    | (summary_df["WindowCheckOK"] == False)
    | ((summary_df["ExpectedTotalAfterVol"].notna()) & (summary_df["TotalGapAfterVol"] != 0))
    | (summary_df["CountsMatchAfterVol"] == False)
)
problem_df = summary_df[problem_mask].copy()

print("\nPotential issues (if any):")
if problem_df.empty:
    print("No issues detected. All datasets passed split checks.")
else:
    display(problem_df)

print("\n" + "=" * 90)
print("Done. You now validated split behavior for ALL datasets in DATASET_FILES.")

Checking 9 dataset(s)...
Split summary across all datasets:


,Dataset,OriginalClosePoints,AfterDateFilterPoints,ExpectedAfterPreprocess,Train,Val,Test,ActualTotal,ExpectedTrain,ExpectedVal,ExpectedTestRaw,ExpectedTestAfterVol,ExpectedTotalRaw,ExpectedTotalAfterVol,TotalGapAfterVol,CountsMatchAfterVol,NaNIssues,WindowCheckOK,WindowIssues
0,DAX_40,4567,4059,3999,1983,1104,912,3999,1983,1104,972,912,4059,3999,0,True,0,True,
1,EuroNext_100,4610,4099,4039,2005,1115,919,4039,2005,1115,979,919,4099,4039,0,True,0,True,
2,IBEX_35,4607,4100,4040,1815,1362,863,4040,1815,1362,923,863,4100,4040,0,True,0,True,
3,KOSPI_index,4435,3934,3874,1944,1109,821,3874,1944,1109,881,821,3934,3874,0,True,0,True,
4,Nikkei_225,4400,3913,3853,1677,1174,1002,3853,1677,1174,1062,1002,3913,3853,0,True,0,True,
5,SMI,4533,4023,3963,1987,1135,841,3963,1987,1135,901,841,4023,3963,0,True,0,True,
6,VN30_INDEX,4242,3992,3932,1971,1096,865,3932,1971,1096,925,865,3992,3932,0,True,0,True,
7,VN_INDEX,4488,3992,3932,1964,1103,865,3932,1964,1103,925,865,3992,3932,0,True,0,True,
8,snp500,4529,4024,3964,1988,1109,867,3964,1988,1109,927,867,4024,3964,0,True,0,True,



Potential issues (if any):
No issues detected. All datasets passed split checks.

Done. You now validated split behavior for ALL datasets in DATASET_FILES.


In [7]:
import pandas as pd

all_results = []
all_predictions = []

for dataset_file in DATASET_FILES:
    per_dataset_output = RESULTS_DIR / f"model_results__{dataset_file.stem}.csv"
    per_dataset_predictions = RESULTS_DIR / "predictions.csv"

    print(f"Calling run_benchmark on: {dataset_file}")
    print(f"log_progress: {LOG_PROGRESS}")

    per_result_df = run_benchmark(
        dataset_dir=dataset_file,
        seq_len=SEQ_LEN,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        output_csv=per_dataset_output,
        device=DEVICE,
        num_workers=NUM_WORKERS,
        log_progress=LOG_PROGRESS,
        split_mode=SPLIT_MODE,
        date_start=DATE_START,
        date_end=DATE_END,
    )

    assert per_dataset_output.exists(), f"Missing output file: {per_dataset_output}"
    assert per_dataset_predictions.exists(), f"Missing output file: {per_dataset_predictions}"

    per_predictions_df = pd.read_csv(per_dataset_predictions)
    all_results.append(per_result_df)
    all_predictions.append(per_predictions_df)

results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()

results_df.to_csv(OUTPUT_CSV, index=False)
predictions_df.to_csv(OUTPUT_PREDICTIONS_CSV, index=False)

print(f"Output metrics saved: {OUTPUT_CSV}")
print(f"Output predictions saved: {OUTPUT_PREDICTIONS_CSV}")
print(f"Output metrics rows: {len(results_df)}")
print(f"Output predictions rows: {len(predictions_df)}")

print("\nMetrics preview:")
display(results_df.head())

print("\nPredictions preview:")
display(predictions_df.head())

Calling run_benchmark on: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\dataset\VN30_INDEX.csv
log_progress: True
[01:41:56] Benchmark started | device=cuda:0 | datasets=1 | seeds=5 | horizons=[1, 3, 5, 10, 21]
[01:41:56] Dataset start: VN30_INDEX.csv
Original data: 4242 points, range 2009-01-05 to 2025-12-31
After date filter [2010-01-01 00:00:00, 2025-12-31 00:00:00]: 3992 points, range 2010-01-04 to 2025-12-31
After rolling volatility (window=60): 3932 return points
Using split mode: fixed_counts
Dataset: VN30_INDEX
Expected split counts: Train=1971, Val=1096, Test=925 (total=3992)
Actual returns points: 3932

Final split (returns/volatility space):
  Train: 1971 points
  Val:   1096 points
  Test:  865 points
  Total: 3932 points

SPLIT REPORT
Date range: 2010-01-01 to 2025-12-31
Split mode: fixed_counts
----------------------------------------------------------------------
Train:   1971 samples (50.1%)
Val:     1096 samples (27.9%)
Test:     865 samples (22.0%)
--------------------

,Dataset,Seed,Horizon,Model,MAE,MSE,QLIKE,Violation_Rate,Kupiec_LR,Kupiec_p,LR_Ind,N_eval,Seq_len
0,VN30_INDEX,42,1,GARCH,21.721419,884.557168,10.382322,0.0,NaN,NaN,NaN,844,60
1,VN30_INDEX,42,3,GARCH,21.714914,884.375882,10.381978,0.0,NaN,NaN,NaN,844,60
2,VN30_INDEX,42,5,GARCH,21.716385,884.355743,10.380390,0.0,NaN,NaN,NaN,844,60
3,VN30_INDEX,42,10,GARCH,21.803667,888.449645,10.379645,0.0,NaN,NaN,NaN,844,60
4,VN30_INDEX,42,21,GARCH,21.870418,892.551059,10.391668,0.0,NaN,NaN,NaN,844,60



Predictions preview:


,time,dataset,model,horizon,True_Volatility,Pred_Volatility,time_train
0,2022-07-20 00:00:00,VN30_INDEX,Autoformer,1,110.122873,140.158196,15.246088
1,2022-07-21 00:00:00,VN30_INDEX,Autoformer,1,111.355678,143.158705,15.246088
2,2022-07-22 00:00:00,VN30_INDEX,Autoformer,1,109.244760,145.537368,15.246088
3,2022-07-25 00:00:00,VN30_INDEX,Autoformer,1,109.426860,147.152373,15.246088
4,2022-07-26 00:00:00,VN30_INDEX,Autoformer,1,101.897170,151.563714,15.246088


In [8]:
import pandas as pd

metrics_df = pd.read_csv(OUTPUT_CSV)
print(f"Loaded metrics CSV: {OUTPUT_CSV}")
print(f"Rows: {len(metrics_df)}")
print(f"Columns: {list(metrics_df.columns)}")

metrics_summary_df = (
    metrics_df.groupby(["Dataset", "Model", "Horizon"], as_index=False)[
        ["MAE", "MSE", "QLIKE", "Violation_Rate", "Kupiec_LR", "Kupiec_p", "LR_Ind"]
    ]
    .mean()
    .sort_values(["Dataset", "Model", "Horizon"])
)

metrics_summary_df.head(30)

Loaded metrics CSV: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\AAAI24_GARCH_NN_Reproduction\experiments\results\model_results_fixed_split_data.csv
Rows: 40
Columns: ['Dataset', 'Seed', 'Horizon', 'Model', 'MAE', 'MSE', 'QLIKE', 'Violation_Rate', 'Kupiec_LR', 'Kupiec_p', 'LR_Ind', 'N_eval', 'Seq_len']


,Dataset,Model,Horizon,MAE,MSE,QLIKE,Violation_Rate,Kupiec_LR,Kupiec_p,LR_Ind
0,VN30_INDEX,Autoformer,1,0.269652,0.115249,1.397047,0.115741,58.290483,2.264855e-14,2.926712
1,VN30_INDEX,Autoformer,3,0.708103,0.565407,1.685096,0.140371,101.666025,0.000000e+00,146.005032
2,VN30_INDEX,Autoformer,5,1.261916,1.645414,2.033929,0.133721,88.798597,0.000000e+00,237.667899
3,VN30_INDEX,Autoformer,10,2.284212,5.275519,2.605587,0.122807,69.101690,1.110223e-16,259.208673
4,VN30_INDEX,Autoformer,21,3.845539,14.955439,3.283189,0.122038,66.947198,3.330669e-16,382.625024
5,VN30_INDEX,FI-GARCH,1,0.276353,0.144003,1.398256,0.107639,46.101696,1.122713e-11,2.787582
6,VN30_INDEX,FI-GARCH,3,0.885479,1.147400,1.725190,0.146172,113.225951,0.000000e+00,126.872641
7,VN30_INDEX,FI-GARCH,5,1.486706,2.866074,2.081668,0.131395,84.535000,0.000000e+00,238.939542
8,VN30_INDEX,FI-GARCH,10,2.597200,8.211354,2.658355,0.120468,65.232054,6.661338e-16,260.723809
9,VN30_INDEX,FI-GARCH,21,4.305750,22.004027,3.341785,0.118483,61.220563,5.107026e-15,417.788849


In [ ]:
import pandas as pd

# Sanity check for new multi-horizon logic:
# 1) Horizon set must be exactly [1, 3, 5, 10, 21]
# 2) N_eval should be fixed across horizons for each (Dataset, Model)
metrics_check_df = pd.read_csv(OUTPUT_CSV)
expected_horizons = {1, 3, 5, 10, 21}
found_horizons = set(metrics_check_df["Horizon"].dropna().astype(int).unique().tolist())

print("Found horizons:", sorted(found_horizons))
assert found_horizons == expected_horizons, (
    f"Unexpected horizons. expected={sorted(expected_horizons)}, found={sorted(found_horizons)}"
)

n_eval_by_h = (
    metrics_check_df.groupby(["Dataset", "Model", "Horizon"], as_index=False)["N_eval"]
    .first()
    .sort_values(["Dataset", "Model", "Horizon"])
    .reset_index(drop=True)
)

n_eval_consistency = (
    n_eval_by_h.groupby(["Dataset", "Model"], as_index=False)["N_eval"]
    .nunique()
    .rename(columns={"N_eval": "N_eval_unique_count"})
    .sort_values(["Dataset", "Model"])
)

bad_pairs = n_eval_consistency[n_eval_consistency["N_eval_unique_count"] != 1]

if bad_pairs.empty:
    print("OK: N_eval is fixed across horizons for all Dataset-Model pairs.")
else:
    print("WARNING: Some Dataset-Model pairs have non-fixed N_eval across horizons.")
    display(bad_pairs)

display(n_eval_by_h.head(20))

import pandas as pd

predictions_df = pd.read_csv(OUTPUT_PREDICTIONS_CSV)
required_columns = [
    "time",
    "dataset",
    "model",
    "horizon",
    "True_Volatility",
    "Pred_Volatility",
    "time_train",
]
missing_columns = [c for c in required_columns if c not in predictions_df.columns]
assert not missing_columns, f"Missing columns in predictions: {missing_columns}"

print(f"Loaded predictions CSV: {OUTPUT_PREDICTIONS_CSV}")
print(f"Rows: {len(predictions_df)}")
print(f"Columns: {list(predictions_df.columns)}")

predictions_h1_df = predictions_df[predictions_df["horizon"] == 1].copy()
print(f"Rows with horizon=1: {len(predictions_h1_df)}")

predictions_h1_df.head(30)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

#plot_df = predictions_df.copy()
plot_df = pd.read_csv(OUTPUT_PREDICTIONS_CSV)
plot_df["time"] = pd.to_datetime(plot_df["time"], errors="coerce")
plot_df = plot_df.dropna(subset=["time"]).sort_values("time")

dataset_options = sorted(plot_df["dataset"].unique().tolist())
model_options = sorted(plot_df["model"].unique().tolist())
horizon_options = sorted(plot_df["horizon"].dropna().unique().astype(int).tolist())

if not dataset_options or not model_options or not horizon_options:
    raise ValueError("No valid data available for interactive visualization")

dataset_selector = widgets.ToggleButtons(
    options=dataset_options,
    description="Dataset:",
)
model_selector = widgets.ToggleButtons(
    options=model_options,
    description="Model:",
)
horizon_selector = widgets.ToggleButtons(
    options=horizon_options,
    description="Horizon:",
    value=1 if 1 in horizon_options else horizon_options[0],
)

def _plot_selected(dataset, model, horizon):
    vis_df = plot_df[
        (plot_df["dataset"] == dataset)
        & (plot_df["model"] == model)
        & (plot_df["horizon"] == horizon)
    ].copy()
    vis_df = vis_df.sort_values("time")

    if vis_df.empty:
        print(f"No rows for dataset={dataset}, model={model}, horizon={horizon}")
        return

    plt.figure(figsize=(14, 5))
    plt.plot(
        vis_df["time"],
        vis_df["True_Volatility"],
        label="True Volatility",
        linewidth=2,
    )
    plt.plot(
        vis_df["time"],
        vis_df["Pred_Volatility"],
        label="Predicted Volatility",
        linewidth=2,
    )
    plt.title(f"True vs Predicted Volatility | {dataset} | {model} | horizon={horizon}")
    plt.xlabel("Time")
    plt.ylabel("Volatility")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

ui = widgets.VBox([
    dataset_selector,
    model_selector,
    horizon_selector,
])
out = widgets.interactive_output(
    _plot_selected,
    {
        "dataset": dataset_selector,
        "model": model_selector,
        "horizon": horizon_selector,
    },
)

display(ui, out)

Output()